# TurboQuant — KV Cache Compression Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OnlyTerp/turboquant/blob/master/notebooks/demo.ipynb)
[![GitHub](https://img.shields.io/github/stars/OnlyTerp/turboquant?style=social)](https://github.com/OnlyTerp/turboquant)

First open-source implementation of [TurboQuant](https://arxiv.org/abs/2504.19874) (ICLR 2026).

**What this notebook does:**
1. Installs TurboQuant
2. Loads TinyLlama-1.1B (fits in Colab free tier)
3. Compresses KV cache at 2.5-bit and 3.5-bit modes
4. Measures cosine similarity, compression ratio, and generation quality

No GPU required (CPU works), but GPU is ~10x faster.

In [ ]:
!pip install -q torch transformers
!pip install -q git+https://github.com/OnlyTerp/turboquant.git

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

import sys, os
# Import from installed package
from src.cache import (
    TurboQuantConfig,
    turboquant_encode_internal,
    polarquant_decode,
    compression_ratio_fp16,
    memory_bytes_per_vector,
    N_OUTLIER_CHANNELS,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

## Load Model

In [ ]:
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model = model.to(device)
model.eval()

# Detect head_dim
with torch.no_grad():
    test = tokenizer('test', return_tensors='pt').to(device)
    out = model(**test, use_cache=True)
    pkv = out.past_key_values
    if hasattr(pkv, 'layers'):
        kv0 = (pkv.layers[0].keys, pkv.layers[0].values)
    elif hasattr(pkv, 'key_cache'):
        kv0 = (pkv.key_cache[0], pkv.value_cache[0])
    else:
        kv0 = (pkv[0][0], pkv[0][1])
    head_dim = kv0[0].shape[-1]
    n_kv_heads = kv0[0].shape[1]
    print(f'head_dim={head_dim}, kv_heads={n_kv_heads}, layers={model.config.num_hidden_layers}')

## Compress & Decompress KV Cache

In [ ]:
def extract_kv(past_kv):
    if hasattr(past_kv, 'layers'):
        return [(l.keys, l.values) for l in past_kv.layers]
    if hasattr(past_kv, 'key_cache'):
        return [(past_kv.key_cache[i], past_kv.value_cache[i]) for i in range(len(past_kv.key_cache))]
    return [(item[0], item[1]) for item in past_kv]

def compress_layer(k, v, layer_idx, config):
    batch, n_heads, seq_len, hd = k.shape
    new_k, new_v = torch.zeros_like(k), torch.zeros_like(v)
    for h in range(n_heads):
        rotation = config.make_rotation(layer_idx, h)
        S = config.make_qjl_matrix(layer_idx, h)
        kf = k[0, h].float().cpu()
        vf = v[0, h].float().cpu()
        mixed = config.get_mixed_config(layer_idx, h, kf)
        kc = turboquant_encode_internal(kf, config.codebook, rotation, S, mixed=mixed)
        vc = turboquant_encode_internal(vf, config.codebook, rotation, S, mixed=mixed)
        new_k[0, h] = polarquant_decode(kc.pq)[..., :hd].to(k.dtype)
        new_v[0, h] = polarquant_decode(vc.pq)[..., :hd].to(v.dtype)
    return new_k, new_v

def rebuild_cache(kv_tuple):
    from transformers.cache_utils import DynamicCache
    cache = DynamicCache()
    for i, (k, v) in enumerate(kv_tuple):
        cache.update(k, v, i)
    return cache

In [ ]:
import time

modes = [
    {'name': '2.5-bit', 'b_mse': 2, 'b_outlier': 3},
    {'name': '3.5-bit', 'b_mse': 3, 'b_outlier': 4},
]

prompts = [
    'The capital of France is',
    'In quantum physics, the uncertainty principle states that',
    'def fibonacci(n):\n    if n <= 1:\n        return n\n    return',
]

for mode in modes:
    print(f"\n{'='*60}")
    cr = compression_ratio_fp16(head_dim, mode['b_mse'], True, min(N_OUTLIER_CHANNELS, head_dim), mode['b_outlier'])
    print(f"  MODE: {mode['name']} | Compression: {cr:.1f}x vs FP16")
    print(f"{'='*60}")

    config = TurboQuantConfig(
        d=head_dim, b_mse=mode['b_mse'], device=torch.device('cpu'),
        mixed_precision=True, n_outlier=min(N_OUTLIER_CHANNELS, head_dim),
        b_outlier=mode['b_outlier'], use_online_codebook=True,
    )

    for pi, prompt in enumerate(prompts):
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**inputs, use_cache=True)
            logits_normal = out.logits[:, -1, :]
            kv_orig = extract_kv(out.past_key_values)

        t0 = time.time()
        kv_recon = [compress_layer(k, v, li, config) for li, (k, v) in enumerate(kv_orig)]
        ms = (time.time() - t0) * 1000

        with torch.no_grad():
            cache = rebuild_cache(kv_recon)
            out_c = model(input_ids=inputs.input_ids[:, -1:], past_key_values=cache, use_cache=False)
            logits_c = out_c.logits[:, -1, :]

        cos = F.cosine_similarity(logits_normal.float().reshape(1,-1), logits_c.float().reshape(1,-1)).item()
        match = (logits_normal.argmax(-1) == logits_c.argmax(-1)).item()
        pred_n = tokenizer.decode(logits_normal.argmax(-1)[0])
        pred_c = tokenizer.decode(logits_c.argmax(-1)[0])

        print(f"  P{pi+1}: cos={cos:.4f} | {'MATCH' if match else 'DIFFER'} | {pred_n.strip()} vs {pred_c.strip()} | {ms:.0f}ms")

## Generation Comparison

In [ ]:
prompt = 'In 1969, humans first'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    normal_out = model.generate(inputs.input_ids, max_new_tokens=40, do_sample=False)
normal_text = tokenizer.decode(normal_out[0], skip_special_tokens=True)
print(f'Normal:     {normal_text}')

# Generate with TurboQuant-compressed KV cache (3.5-bit)
config = TurboQuantConfig(
    d=head_dim, b_mse=3, device=torch.device('cpu'),
    mixed_precision=True, n_outlier=min(N_OUTLIER_CHANNELS, head_dim),
    b_outlier=4, use_online_codebook=True,
)

with torch.no_grad():
    out = model(input_ids=inputs.input_ids, use_cache=True)
    kv_orig = extract_kv(out.past_key_values)
    kv_recon = [compress_layer(k, v, li, config) for li, (k, v) in enumerate(kv_orig)]
    cache = rebuild_cache(kv_recon)

    generated_ids = inputs.input_ids.clone()
    logits = out.logits[:, -1, :]
    next_token = logits.argmax(-1, keepdim=True)
    generated_ids = torch.cat([generated_ids, next_token], dim=-1)

    for _ in range(39):
        out = model(input_ids=generated_ids[:, -1:], past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        next_token = out.logits[:, -1, :].argmax(-1, keepdim=True)
        if next_token.item() == tokenizer.eos_token_id:
            break
        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

tq_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f'TurboQuant: {tq_text}')

---

**Paper:** [TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate](https://arxiv.org/abs/2504.19874) (ICLR 2026)

**GitHub:** [OnlyTerp/turboquant](https://github.com/OnlyTerp/turboquant)